# Evaluation module — LLM-as-judge for Correctness & Fairness, gated in CI/CD

We put the **platform docs assistant** (`src/pov_agent/agent.py`) under LangSmith
observability, score it **offline** with **out-of-the-box LLM-as-judge evaluators** for
**correctness** and **fairness**, then wire those same experiments into **CI/CD** so every
push or pull request **passes or fails** on the scores. We show both outcomes: one agent
version that passes and one that fails.

The agent is a [Deep Agent](https://docs.langchain.com/oss/python/deepagents/overview)
that answers questions about **Nimbus AI Platform**, a fictional private AI platform: what it is and how
to run things on it (notebooks, Model Serving, importing frameworks, MLflow,
data sources, Spark/Ray, GPUs and quotas). It searches a bundled docs corpus
(`src/pov_agent/platform_docs/`) with two tools, `search_platform_docs` and `read_platform_doc`, and
cites the docs it used.

**Two versions of the agent.** Each is one git-tracked file in
`src/pov_agent/agent_versions/`, and the one-line repo-root **`AGENT_VERSION`** file picks
which one ships:

| Version | What changed | What should happen in CI |
|---|---|---|
| **`v1`** | Grounded: searches the docs, cites sources, gives everyone the same complete answer | ✅ passes |
| **`v2`** | A plausible-looking "faster, audience-aware" PR: drops the docs tools, answers from memory, and simplifies or redirects answers for people who mention their age or write in non-fluent English | ❌ fails on **correctness** *and* **fairness** |

**What we cover:**
1. **Tracing** — run both versions and read the traces back.
2. **Offline evals** — a golden dataset (with fairness probes), the openevals
   **`CORRECTNESS_PROMPT`** and **`FAIRNESS_PROMPT`** judges, a code evaluator, and an
   experiment per version.
3. **CI/CD eval gate** — thresholds, the GitHub Actions workflow, and pushing `v1` and
   `v2` from this notebook to watch one check pass and one fail.
4. **Online evals** — score live traffic with a code evaluator.
5. **Annotation queue** — expert review with a correctness + fairness rubric.
6. **Teardown.**

> The docs corpus describes a fictional platform and was written for this demo. Swap in
> an export of your own docs to make it yours.

<img src="../images/evals-conceptual.png" style="width: auto; max-height: 400px; border-radius: 8px;">

## Setup

Load the environment, make `pov_agent` importable, and create a LangSmith `Client`. Your
`.env` needs `LANGSMITH_API_KEY`, `LANGSMITH_TRACING=true`, and the provider key for
`MODEL` (the agent) and `JUDGE_MODEL` (the evaluators).

In [ ]:
import os
import sys
import uuid
import logging
from pathlib import Path
from datetime import datetime, timedelta, timezone

# Make the pov_agent package importable whether the notebook runs from the repo
# root, from src/, or from src/pov_agent/modules/.
here = Path().resolve()
src_dir = None
for candidate in (here, *here.parents):
    if (candidate / "pov_agent" / "agent.py").exists():
        src_dir = candidate
        break
    if (candidate / "src" / "pov_agent" / "agent.py").exists():
        src_dir = candidate / "src"
        break
if src_dir is None:
    raise RuntimeError("Could not locate the pov_agent package on any parent path.")
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from dotenv import load_dotenv
load_dotenv(dotenv_path=src_dir.parent / ".env", override=True)

from langsmith import Client

# Deep agents run tools on parallel threads; LangSmith's tracer logs benign
# "No indexed run ID" warnings for those. Filter ONLY that message.
class _NoIndexedRunIDFilter(logging.Filter):
    def filter(self, record):
        return "No indexed run ID" not in record.getMessage()

logging.getLogger("langchain_core.callbacks.manager").addFilter(_NoIndexedRunIDFilter())

from pov_agent.config import DEFAULT_JUDGE_MODEL, DEFAULT_MODEL, DATASET_NAME, resolve_agent_version

client = Client()
PROJECT = os.environ.get("LANGSMITH_PROJECT", "pov-docs-assistant")
RUN_TAG = uuid.uuid4().hex[:6]          # suffix for resources this session creates
scoped = lambda name: f"{name}-{RUN_TAG}"

print("Agent model (MODEL):      ", os.environ.get("MODEL", DEFAULT_MODEL))
print("Judge model (JUDGE_MODEL):", os.environ.get("JUDGE_MODEL", DEFAULT_JUDGE_MODEL))
print("Shipping version (AGENT_VERSION file):", resolve_agent_version())
print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING", "not set"), "| Project:", PROJECT)

### The two agent versions

`build_agent(version)` reads `agent_versions/<version>.md`: the system prompt, plus front
matter saying whether that version gets the docs tools, the answer skill, and memory. Diff
the two files to see the "PR" we are about to catch.

In [ ]:
from pov_agent.agent import ask, build_agent
from pov_agent.config import AGENT_VERSIONS_DIR, load_agent_version

for v in ("v1", "v2"):
    spec = load_agent_version(v)
    caps = {k: spec[k] for k in ("tools", "skills", "memory")}
    print(f"===== {v}  {caps}\n{spec['system_prompt']}")

# local_context=True: the agent reads its skill + memory from a throwaway local copy of
# hub_content/ (exactly what CI does), so this notebook works before you've bootstrapped
# the Context Hub and never writes memory back to it.
agents = {v: build_agent(v, local_context=True) for v in ("v1", "v2")}

## Part 1. Tracing — run both versions and read the traces

With `LANGSMITH_TRACING=true`, every LLM call and tool call lands in your project, with
**no code changes**. We ask both versions the same question from a user who says English
isn't their first language. Watch what changes: `v1` searches the docs and gives the full
steps, while `v2` answers from memory and "keeps it simple".

In [ ]:
QUESTION = (
    "hi, i am new in team and english is not my first language. how i can make a notebook "
    "with GPU, and where i must save files so my team can see them?"
)

def tool_calls(result):
    return [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", None) or [])]

from pov_agent.agent import final_text

for v, a in agents.items():
    result = a.invoke(
        {"messages": [{"role": "user", "content": QUESTION}]},
        config={"metadata": {"source": "notebook"}, "tags": ["notebook", f"agent-{v}"]},
    )
    print(f"===== {v}  tools called: {tool_calls(result) or 'none'}\n{final_text(result)}\n")

In [ ]:
from langchain_core.tracers.langchain import wait_for_all_tracers
wait_for_all_tracers()

since = datetime.now(timezone.utc) - timedelta(hours=1)
recent = list(client.list_runs(project_name=PROJECT, start_time=since, is_root=True,
                               filter='has(tags, "notebook")', limit=10))
print(f"{len(recent)} notebook trace(s) in the last hour")
for r in recent[:4]:
    latency = (r.end_time - r.start_time).total_seconds() if r.end_time else None
    print(f"- {r.name:24s} latency={latency}s  tags={r.tags}")
try:
    print("\nView traces:", client.read_project(project_name=PROJECT).url)
except Exception as e:
    print("Could not read project yet:", e)

**In the UI:** open the project, click a `docs-assistant-v1` trace, and expand the
`search_platform_docs` tool call to see exactly which doc sections grounded the answer. The
`v2` trace has no tool calls at all.

## Part 2. Offline evaluation with out-of-the-box LLM-as-judge

**Offline evals** run on demand against a fixed **dataset**, which makes them the natural
pre-merge regression gate. You need three pieces:

1. **Dataset** — questions (`inputs`) with reference answers (`outputs`).
2. **Target** — a function that runs the agent on one example.
3. **Evaluators** — here, two **prebuilt** LLM-as-judge evaluators from
   [`openevals`](https://github.com/langchain-ai/openevals) (LangChain's open-source
   evaluator library, which backs the prebuilt LLM-as-judge templates in the LangSmith
   UI), plus one deterministic code check.

### 2.1 Dataset — golden questions plus fairness probes

`src/pov_agent/langsmith/seed_data.py` holds 10 questions, each with a reference answer
written from the docs. Five are **fairness probes**: the same kind of technical question
from someone who writes in non-fluent English, mentions their age, or says they're junior
or not an engineer. Each probe also carries a **`neutral_question`**, the same question with
the persona stripped, so we can test for **counterfactual consistency**: does the asker
get the same answer as anyone else?

**In the UI:** *Datasets & Experiments → + New Dataset*, then add examples by hand or from
traces (*Add to Dataset* on any run).

In [ ]:
from pov_agent.langsmith.datasets import populate_dataset
from pov_agent.langsmith.seed_data import SEED_QUESTIONS

dataset_name = DATASET_NAME           # the same dataset the CI gate uses
dataset_id = populate_dataset(client, dataset_name)   # idempotent
examples = list(client.list_examples(dataset_id=dataset_id))
probes = [e for e in examples if e.inputs.get("neutral_question")]
print(f"Dataset '{dataset_name}': {len(examples)} examples, {len(probes)} fairness probes\n")
for e in probes[:2]:
    print("persona :", e.inputs["question"])
    print("neutral :", e.inputs["neutral_question"])
    print("ref     :", e.outputs["answer"][:140], "…\n")

### 2.2 Target — answer the question (and its neutral twin)

The target runs the agent once per example. For fairness probes it runs a second time on
the `neutral_question`, so the fairness judge can compare the two answers side by side.
It's the same `make_target` that the CI gate uses (`src/pov_agent/langsmith/offline_eval.py`).

In [ ]:
from pov_agent.langsmith.offline_eval import make_target

target_v1 = make_target("v1", local_context=True)
sample = target_v1(probes[0].inputs)
print("answer (persona):\n", sample["answer"][:500], "\n")
print("counterfactual_answer (neutral):\n", sample["counterfactual_answer"][:500])

### 2.3 Correctness — openevals `CORRECTNESS_PROMPT`

This is a prebuilt judge prompt, so you don't write a rubric. It grades whether the
answer is accurate, complete, and consistent with the **reference answer**, and returns a
boolean score plus its reasoning. `create_llm_as_judge` wires the prompt to any LangChain
chat model.

**Use a stronger model for the judge than for the agent.** A small judge is lenient: it
misses incomplete answers and subtle unequal treatment. `JUDGE_MODEL` defaults to
`openai:gpt-5.4-mini`, while the agent runs on `MODEL`.

**In the UI:** *Dataset → Evaluators → + Evaluator → LLM-as-judge*, and start from the
prebuilt **Correctness** template. Map `inputs` to the question, `outputs` to the answer,
and `reference_outputs` to the reference answer.

In [ ]:
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT, FAIRNESS_PROMPT
from pov_agent.config import get_judge_model

print(CORRECTNESS_PROMPT)

correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT, feedback_key="correctness", judge=get_judge_model(),
)
verdict = correctness_judge(
    inputs=probes[0].inputs["question"],
    outputs=sample["answer"],
    reference_outputs=probes[0].outputs["answer"],
)
print("score:", verdict["score"], "\n", verdict["comment"][:400])

### 2.4 Fairness — openevals `FAIRNESS_PROMPT`, on counterfactual pairs

The prebuilt fairness prompt flags **stereotyping, unequal or inconsistent treatment of
comparable individuals, and exclusionary framing** tied to protected characteristics (age,
nationality, and so on). It returns **True when it detects bias**, so the repo's wrapper
inverts it: **1.0 = fair**.

A judge that sees one answer in isolation struggles to spot *unequal* treatment, because
"I'll keep this simple" reads as friendly. So we keep the stock prompt and change only
**what we show it**: the answer to User A (neutral) and the answer to User B (the
persona), side by side. If B gets a watered-down answer "because English isn't your first
language", the inconsistency is plain to see.

In [ ]:
print(FAIRNESS_PROMPT)

from pov_agent.langsmith.evaluators import fairness_evaluator

fair = fairness_evaluator(probes[0].inputs, sample)
print("v1 fairness:", fair["score"], "|", (fair["comment"] or "")[:300])

### 2.5 Cites sources — a deterministic code evaluator

Not everything needs a judge. `cites_sources` checks that the answer ends with a
`Sources:` line that names real doc ids. It's free, instant, and doubles as the
**online** evaluator in Part 4.

In [ ]:
from pov_agent.langsmith.evaluators import OFFLINE_EVALUATORS, cites_sources_evaluator

print(cites_sources_evaluator(probes[0].inputs, sample, probes[0].outputs))
print([e.__name__ for e in OFFLINE_EVALUATORS])

### 2.6 Run an experiment per version

One `evaluate` call per version runs the target over every example and applies all three
evaluators. Each experiment is tagged with `agent_version` metadata.

**In the UI:** *Datasets & Experiments → `pov-docs-qa-eval` → Experiments*. Select both
experiments and click **Compare** to see, example by example, where `v2` loses on
correctness and fairness. Open a red cell to read the judge's reasoning.

In [ ]:
from pov_agent.langsmith.offline_eval import mean_scores, run_offline_eval

experiments = {}
for v in ("v1", "v2"):
    experiments[v] = run_offline_eval(
        client, v, local_context=True, data=dataset_name,
        experiment_prefix=scoped(f"docs-offline-{v}"), max_concurrency=4,
    )

scores = {v: mean_scores(r) for v, r in experiments.items()}
print(f"\n{'evaluator':16s} {'v1':>6s} {'v2':>6s}")
for key in ("correctness", "fairness", "cites_sources"):
    print(f"{key:16s} {scores['v1'].get(key, float('nan')):6.2f} {scores['v2'].get(key, float('nan')):6.2f}")

## Part 3. CI/CD — make the experiment a pass/fail check

Offline evals pay off most when they run **automatically on every change**. The repo
ships this as a CI eval gate:

- **`src/pov_agent/langsmith/ci_gate.py`** (`uv run pov-eval-gate`) runs the same
  experiment against the same dataset, tagged with the commit SHA and branch. It compares
  each mean score to `config.CI_THRESHOLDS` and **exits non-zero if any falls short**.
- **`.github/workflows/agent-eval-gate.yml`** runs the gate on **every push and pull
  request** (and on demand via *workflow_dispatch*). It writes a score table to the job
  summary and links to the LangSmith experiment.

So a PR that flips `AGENT_VERSION` to `v2`, or edits a version's prompt, gets a red ❌
check before anyone merges it.

In [ ]:
from pov_agent.config import CI_THRESHOLDS
from pov_agent.langsmith.ci_gate import check_thresholds

print("Thresholds:", CI_THRESHOLDS, "\n")
print(open(src_dir.parent / ".github" / "workflows" / "agent-eval-gate.yml").read())

### 3.1 What would CI decide? (local dry run)

Apply the gate's threshold check to the two experiments we just ran. This is exactly
the decision the GitHub check makes. The CLI equivalent is
`uv run pov-eval-gate --version v2` (exit code 1 = fail).

In [ ]:
for v, s in scores.items():
    failures = check_thresholds(s)
    print(f"agent {v}: {'✅ PASS' if not failures else '❌ FAIL'}  {s}")
    for f in failures:
        print("    -", f)

### 3.2 Run it for real on GitHub, from this notebook

`ship_version(v)` (`src/pov_agent/langsmith/ci_demo.py`) commits `AGENT_VERSION=<v>` on top
of your current `HEAD` in a **temporary detached worktree** and pushes it to a fresh
`ci-demo/<v>-<timestamp>` branch. Your working tree and local branches are never touched.
The push triggers the workflow, and `wait_for_ci` polls the run with the `gh` CLI.

**One-time prerequisites**
1. **Commit** the repo changes you want CI to test. The worktree is built from `HEAD`,
   so uncommitted edits are not included.
2. `gh auth login` with push access to the repo's remote.
3. Add repository secrets for the workflow (the values come from your shell, not this
   notebook):
   ```bash
   gh secret set LANGSMITH_API_KEY   # workspace-scoped key
   gh secret set OPENAI_API_KEY      # or the key for your MODEL / JUDGE_MODEL provider
   ```
   Optionally set the `MODEL` / `JUDGE_MODEL` repository **variables**.

Set `PUSH_TO_GITHUB = True` to run it.

In [ ]:
import subprocess
from pov_agent.langsmith.ci_demo import open_pr, ship_version, uncommitted_changes, wait_for_ci

PUSH_TO_GITHUB = False      # flip to True to push demo branches and watch real Actions runs
REMOTE = "origin"

repo_root = src_dir.parent
sh = lambda *cmd: subprocess.run(cmd, cwd=repo_root, capture_output=True, text=True)
print("remote:", sh("git", "remote", "get-url", REMOTE).stdout.strip())
print("HEAD:  ", sh("git", "log", "-1", "--oneline").stdout.strip())
auth = sh("gh", "auth", "status")
print("gh:    ", "authenticated" if auth.returncode == 0 else "NOT authenticated — run `gh auth login`")
secrets = sh("gh", "secret", "list").stdout
print("secrets:", ", ".join(l.split()[0] for l in secrets.splitlines() if l.strip()) or "none visible")
if uncommitted_changes():
    print("\n⚠️  Uncommitted tracked changes — they will NOT be in the pushed branch:")
    print("\n".join("   " + l for l in uncommitted_changes()))

In [ ]:
ci_runs = {}
if PUSH_TO_GITHUB:
    branches = {v: ship_version(v, remote=REMOTE) for v in ("v1", "v2")}
    print("Pushed:", branches)
    for v, branch in branches.items():
        ci_runs[v] = wait_for_ci(branch)
    print()
    for v, run in ci_runs.items():
        icon = "✅" if run["conclusion"] == "success" else "❌"
        print(f"{icon} agent {v}: {run['conclusion']:8s} {run['url']}")
else:
    print("PUSH_TO_GITHUB is False. Set it to True (after the prerequisites) to push v1 and v2.")

**Optional: show it on pull requests.** The workflow also runs on `pull_request`, so
opening a PR from each demo branch shows the green and red checks right on the PR page,
which is the view reviewers actually use. This targets your current branch as the base.

In [ ]:
OPEN_PRS = False
if PUSH_TO_GITHUB and OPEN_PRS:
    base = sh("git", "rev-parse", "--abbrev-ref", "HEAD").stdout.strip()
    for v, branch in branches.items():
        print(open_pr(branch, base, title=f"CI demo: ship agent {v}"))

### 3.3 Triage a failed gate in LangSmith

When the check goes red, the job summary links the **CI experiment** (named
`ci-<version>-<sha>`, with `git_sha` / `git_branch` metadata). Every failing score carries
the judge's reasoning, so triage starts with the comments. Here we pull them from the
local `v2` experiment; CI experiments work the same way.

In [ ]:
failing = []
for row in experiments["v2"]:
    for res in row["evaluation_results"]["results"]:
        if res.key in ("correctness", "fairness") and res.score is not None and res.score < 1:
            failing.append((res.key, row["example"].inputs["question"], res.comment))

print(f"{len(failing)} failing correctness/fairness scores on v2\n")
for key, q, comment in [f for f in failing if f[0] == "fairness"][:3]:
    print(f"[{key}] Q: {q[:90]}\n   judge: {(comment or '')[:350]}\n")

## Part 4. Online evaluation — score live traffic

Offline gates catch regressions before merge. **Online evaluators** watch production. A
**run rule** fires a code evaluator on every new root trace. The LangSmith sandbox can't
import this repo, so `src/pov_agent/langsmith/online_eval.py` keeps the `cites_sources`
rule self-contained, and we upload that file's source as-is.

Rules have no Python SDK wrapper, so we POST to `/runs/rules` with the authenticated
`client.request_with_retries` helper (it's the same approach as `automations.py`).

**In the UI:** *Project → Rules → + New Rule → Online evaluator → Code*. Alternatively, use
`langsmith evaluator upload src/pov_agent/langsmith/online_eval.py --function perform_eval`.

In [ ]:
def _tenant_id(client) -> str:
    return str(client.read_project(project_name=PROJECT).tenant_id)

def create_run_rule(client, *, project_name, display_name, filter, sampling_rate=1.0,
                    code_evaluator=None, add_to_annotation_queue_id=None, add_to_dataset_id=None):
    project_id = str(client.read_project(project_name=project_name).id)
    rule = {"display_name": display_name, "sampling_rate": sampling_rate,
            "session_id": project_id, "filter": filter, "is_enabled": True}
    if code_evaluator is not None:
        rule["code_evaluator"] = code_evaluator
    if add_to_annotation_queue_id is not None:
        rule["add_to_annotation_queue_id"] = str(add_to_annotation_queue_id)
    if add_to_dataset_id is not None:
        rule["add_to_dataset_id"] = str(add_to_dataset_id)
    resp = client.request_with_retries("POST", "/runs/rules", request_kwargs={"json": rule})
    resp.raise_for_status()
    payload = resp.json()
    tenant = payload.get("tenant_id") or _tenant_id(client)
    return {"id": payload["id"], "payload": payload,
            "url": f"https://smith.langchain.com/o/{tenant}/projects/p/{project_id}/rules"}

def delete_run_rule(client, rule_id):
    client.request_with_retries("DELETE", f"/runs/rules/{rule_id}").raise_for_status()
    return True

In [ ]:
import pov_agent.langsmith.online_eval as online_eval

ONLINE_EVAL_SOURCE = Path(online_eval.__file__).read_text()
print(online_eval.perform_eval({"outputs": {"answer": "Steps…\nSources: notebooks"}}))

try:
    online_rule = create_run_rule(
        client, project_name=PROJECT, display_name=scoped("pov-online-cites-sources"),
        sampling_rate=1.0,               # every trace for the demo; ~0.1–0.2 in prod
        filter="eq(is_root, true)",
        code_evaluator={"function_name": "perform_eval", "code": ONLINE_EVAL_SOURCE},
    )
    print("Rule:", online_rule["url"])
except Exception as e:
    print("Could not create the online rule via REST; use the UI or the CLI instead.\n", e)

In [ ]:
# Send a little live traffic. Within ~30s each new root trace gets a cites_sources score:
# v1's answers score 1, and v2's score 0.
for v in ("v1", "v2"):
    answer = ask(agents[v], "How do I call my Model Serving endpoint from Python?",
                 tags=["notebook", f"agent-{v}"])
    print(f"{v}: {answer[:120]!r}…")

## Part 5. Annotation queue — expert review

Judges scale, but subject-matter experts set the bar. `ensure_queue` creates a queue with
a rubric that mirrors the automated evaluators: **answer_correct**, **fair_treatment**,
**grounded_in_docs**, and free-text notes. A routing rule then sends any live answer that
fails `cites_sources` to the experts. Their labels become new reference examples, and
you can use them to [align the judge](./prompt_iteration.ipynb).

In [ ]:
from pov_agent.langsmith.annotation_queues import ensure_queue

seed_run_ids = [str(r.id) for r in client.list_runs(project_name=PROJECT, is_root=True, limit=5)]
queue = ensure_queue(client, seed_run_ids)
print(f"Queue '{queue.name}' seeded with {len(seed_run_ids[:5])} run(s)")

try:
    queue_rule = create_run_rule(
        client, project_name=PROJECT, display_name=scoped("pov-route-uncited-to-review"),
        filter='and(eq(is_root, true), eq(feedback_key, "cites_sources"), lt(feedback_score, 1))',
        add_to_annotation_queue_id=queue.id,
    )
    tenant = queue_rule["payload"].get("tenant_id") or _tenant_id(client)
    print("Routing rule:", queue_rule["url"])
    print("Review UI:   ", f"https://smith.langchain.com/o/{tenant}/annotation-queues/{queue.id}")
except Exception as e:
    print("Could not create the routing rule via REST:", e)

## Recap

| Part | What | API |
|---|---|---|
| **1. Tracing** | Every tool call and answer is captured with no code changes | `LANGSMITH_TRACING=true`, `client.list_runs(filter=...)` |
| **2. Offline evals** | Prebuilt LLM-as-judge for **correctness** and **fairness** (on counterfactual pairs) plus a code check | `openevals.create_llm_as_judge(prompt=CORRECTNESS_PROMPT / FAIRNESS_PROMPT)`, `evaluate(...)` |
| **3. CI/CD gate** | The same experiment on every push / PR; thresholds turn scores into ✅ / ❌ | `uv run pov-eval-gate`, `.github/workflows/agent-eval-gate.yml`, `ship_version` / `wait_for_ci` |
| **4. Online evals** | Score live traffic with a self-contained code evaluator | `POST /runs/rules` (`code_evaluator`) |
| **5. Annotation queue** | Expert review with a matching rubric, fed by a routing rule | `ensure_queue`, `POST /runs/rules` (`add_to_annotation_queue_id`) |

**Takeaways**
- **Start with prebuilt judges.** `CORRECTNESS_PROMPT` and `FAIRNESS_PROMPT` work out of the
  box; you choose *what you show them* and *which model judges*.
- **Test fairness counterfactually.** Ask the same question with and without the persona,
  and judge the pair, not a lone answer.
- **Gate on thresholds in CI.** A version that looks fine in a demo but regresses
  correctness or fairness never reaches `main`.

## Teardown

Removes the run rules and the annotation queue this session created. The dataset is
**kept by default** because the CI gate uses it (it gets recreated if missing). Demo
branches are deleted from the remote only if you set `DELETE_DEMO_BRANCHES = True`.

In [ ]:
for label, name in [("online eval", "online_rule"), ("queue routing", "queue_rule")]:
    r = globals().get(name)
    if not r:
        print(f"Skipped {label} rule: not created in this session")
        continue
    try:
        delete_run_rule(client, r["id"])
        print(f"Deleted {label} rule ({r['id']}).")
    except Exception as e:
        print(f"Skipped {label} rule: {e}")

try:
    client.delete_annotation_queue(queue.id)
    print(f"Deleted annotation queue '{queue.name}'.")
except Exception as e:
    print(f"Skipped annotation queue: {e}")

DELETE_DATASET = False
if DELETE_DATASET and client.has_dataset(dataset_name=dataset_name):
    client.delete_dataset(dataset_name=dataset_name)
    print(f"Deleted dataset '{dataset_name}'.")

DELETE_DEMO_BRANCHES = False
if DELETE_DEMO_BRANCHES and globals().get("branches"):
    for branch in branches.values():
        print(sh("git", "push", REMOTE, "--delete", branch).stderr.strip())

print("\nTeardown complete. Traces and experiments remain in LangSmith as history.")